# imports y raíz del proyecto

In [13]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
SOURCE_DIR = DATA_DIR / "source"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SOURCE_DIR:", SOURCE_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
SOURCE_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests


In [14]:
nih_source_dir = SOURCE_DIR / "nih"
manifest_final_path = MANIFESTS_DIR / "manifest_nih_final.csv"
subset_small_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_small.csv"

print("nih_source_dir:", nih_source_dir)
print("manifest_final_path:", manifest_final_path)
print("subset_small_path:", subset_small_path)

print("\n¿Existen?")
print("nih_source_dir ->", nih_source_dir.exists())
print("manifest_final.csv ->", manifest_final_path.exists())
print("nih_subset_small.csv ->", subset_small_path.exists())

nih_source_dir: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih
manifest_final_path: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_final.csv
subset_small_path: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_small.csv

¿Existen?
nih_source_dir -> True
manifest_final.csv -> True
nih_subset_small.csv -> True


# inspeccionar la carpeta fuente NIH

In [15]:
print("Primer nivel dentro de data/source/nih:\n")
for p in sorted(nih_source_dir.iterdir()):
    print("-", p)

Primer nivel dentro de data/source/nih:

- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/ARXIV_V5_CHESTXRAY.pdf
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/BBox_List_2017.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/Data_Entry_2017.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/FAQ_CHESTXRAY.pdf
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/LOG_CHESTXRAY.pdf
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/README_CHESTXRAY.pdf
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/archive.zip
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_002
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_003
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_004
- /mnt/d/Universidad/analitica/proyecto_analiti

# buscar todas las imágenes PNG descomprimidas


In [16]:
png_files = list(nih_source_dir.rglob("*.png"))

print("Cantidad total de PNG encontrados:", len(png_files))
print("\nPrimeros 10 PNG:")
for p in png_files[:10]:
    print(p)

Cantidad total de PNG encontrados: 112120

Primeros 10 PNG:
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_000.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_001.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_002.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000002_000.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_000.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_001.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_002.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_003.png
/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_004.png
/mnt/d/Universidad/analitica/proyecto_an

# construir mapa image_name -> ruta real

In [17]:
image_map = {p.name: str(p.resolve()) for p in png_files}

print("Cantidad de entradas en image_map:", len(image_map))

sample_keys = list(image_map.keys())[:5]
print("\nEjemplos:")
for k in sample_keys:
    print(k, "->", image_map[k])

Cantidad de entradas en image_map: 112120

Ejemplos:
00000001_000.png -> /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_000.png
00000001_001.png -> /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_001.png
00000001_002.png -> /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000001_002.png
00000002_000.png -> /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000002_000.png
00000003_000.png -> /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/nih/images_001/images/00000003_000.png


# cargar el manifiesto final y agregar rutas

In [18]:
nih_final = pd.read_csv(manifest_final_path)

nih_final["file_path"] = nih_final["image_name"].map(image_map)

missing_paths = nih_final["file_path"].isna().sum()

print("Shape:", nih_final.shape)
print("Imágenes sin ruta encontrada:", missing_paths)

nih_final[["image_name", "file_path"]].head()

Shape: (112120, 19)
Imágenes sin ruta encontrada: 0


,image_name,file_path
0,00000001_000.png,/mnt/d/Universidad/analitica/proyecto_analitic...
1,00000001_001.png,/mnt/d/Universidad/analitica/proyecto_analitic...
2,00000001_002.png,/mnt/d/Universidad/analitica/proyecto_analitic...
3,00000002_000.png,/mnt/d/Universidad/analitica/proyecto_analitic...
4,00000005_000.png,/mnt/d/Universidad/analitica/proyecto_analitic...


## Nueva prueba

In [19]:
subset_large_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_large.csv"
nih_subset_large = pd.read_csv(subset_large_path)

nih_subset_large["file_path"] = nih_subset_large["image_name"].map(image_map)

missing_large_paths = nih_subset_large["file_path"].isna().sum()

print("Shape subset large:", nih_subset_large.shape)
print("Rutas faltantes en large:", missing_large_paths)

nih_subset_large[["image_name", "split_final", "file_path"]].head()

Shape subset large: (19000, 19)
Rutas faltantes en large: 0


,image_name,split_final,file_path
0,00022245_021.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
1,00019544_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
2,00009673_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
3,00018103_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
4,00017799_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...


In [20]:
subset_large_with_paths_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_large_with_paths.csv"
nih_subset_large.to_csv(subset_large_with_paths_path, index=False)

print("Guardado en:", subset_large_with_paths_path)

Guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_large_with_paths.csv


# guardar manifiesto con rutas reales

In [21]:
manifest_with_paths_path = MANIFESTS_DIR / "manifest_nih_final_with_paths.csv"
nih_final.to_csv(manifest_with_paths_path, index=False)

print("Guardado en:", manifest_with_paths_path)

Guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_final_with_paths.csv


# cargar subset pequeño y agregar rutas

In [22]:
nih_subset_small = pd.read_csv(subset_small_path)

nih_subset_small["file_path"] = nih_subset_small["image_name"].map(image_map)

missing_subset_paths = nih_subset_small["file_path"].isna().sum()

print("Shape subset:", nih_subset_small.shape)
print("Rutas faltantes en subset:", missing_subset_paths)

nih_subset_small[["image_name", "split_final", "file_path"]].head()

Shape subset: (900, 19)
Rutas faltantes en subset: 0


,image_name,split_final,file_path
0,00022245_021.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
1,00019544_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
2,00009673_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
3,00018103_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
4,00017799_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...


# guardar subset pequeño con rutas

In [23]:
subset_with_paths_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_small_with_paths.csv"
nih_subset_small.to_csv(subset_with_paths_path, index=False)

print("Guardado en:", subset_with_paths_path)

Guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_small_with_paths.csv


In [24]:
print("Archivos generados:")
print("-", manifest_with_paths_path)
print("-", subset_with_paths_path)

print("\n¿Existen?")
print("manifest_nih_final_with_paths.csv ->", manifest_with_paths_path.exists())
print("nih_subset_small_with_paths.csv ->", subset_with_paths_path.exists())

print("\nConteo subset por split:")
print(nih_subset_small["split_final"].value_counts())

Archivos generados:
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_nih_final_with_paths.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_small_with_paths.csv

¿Existen?
manifest_nih_final_with_paths.csv -> True
nih_subset_small_with_paths.csv -> True

Conteo subset por split:
split_final
train    500
val      200
test     200
Name: count, dtype: int64


carga del subset nuevo

In [25]:
subset_baseline_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_baseline.csv"
nih_subset_baseline = pd.read_csv(subset_baseline_path)

nih_subset_baseline["file_path"] = nih_subset_baseline["image_name"].map(image_map)

missing_baseline_paths = nih_subset_baseline["file_path"].isna().sum()

print("Shape subset baseline:", nih_subset_baseline.shape)
print("Rutas faltantes en baseline:", missing_baseline_paths)

nih_subset_baseline[["image_name", "split_final", "file_path"]].head()

Shape subset baseline: (7000, 19)
Rutas faltantes en baseline: 0


,image_name,split_final,file_path
0,00022245_021.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
1,00019544_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
2,00009673_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
3,00018103_001.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...
4,00017799_000.png,train,/mnt/d/Universidad/analitica/proyecto_analitic...


In [26]:
subset_baseline_with_paths_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_baseline_with_paths.csv"
nih_subset_baseline.to_csv(subset_baseline_with_paths_path, index=False)

print("Guardado en:", subset_baseline_with_paths_path)

Guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/nih/subsets/nih_subset_baseline_with_paths.csv
